In [4]:
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Параметры обучения

In [ ]:
METHODS = ("zero", "kaiming", "xavier")
BASE_CHANNELS = 8
BLOCKS_PER_STAGE = 1
EPOCHS = 60
BATCH_SIZE = 128
LR = 0.1
WARMUP_EPOCHS = 10
MILESTONES = (100, 150)
RANK_EVERY = 1
SEED = 24
WORKERS = 2
DATA_DIR = "./data"
OUTPUT_DIR = "./results"

# Модель

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
            if stride != 1 or in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        identity = self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

In [ ]:
class SmallResNet(nn.Module):

    def __init__(self, num_classes=10, base_channels=BASE_CHANNELS):
        super().__init__()
        self.in_channels = base_channels
        self.conv1 = nn.Conv2d(3, base_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(base_channels)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(base_channels, BLOCKS_PER_STAGE, stride=1)
        self.layer2 = self._make_layer(base_channels * 2, BLOCKS_PER_STAGE, stride=2)
        self.layer3 = self._make_layer(base_channels * 4, BLOCKS_PER_STAGE, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(base_channels * 4, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = [BasicBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        layers.extend(BasicBlock(out_channels, out_channels) for _ in range(blocks - 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

In [ ]:
def hadamard(order, *, device, dtype): #матрица адамара нужного размера
    h = torch.ones((1, 1), device=device, dtype=dtype)
    while h.shape[0] < order:
        h = torch.cat(
            (torch.cat((h, h), dim=1), torch.cat((h, -h), dim=1)), dim=0
        )
    return h


@torch.no_grad()
def zero_matrix_(weight) -> None:
    rows, cols = weight.shape
    weight.zero_()
    if rows <= cols:
        weight[:rows, :rows].copy_(
            torch.eye(rows, device=weight.device, dtype=weight.dtype)
        )
        return

    power = math.ceil(math.log2(rows))
    order = 2 ** power
    weight.copy_(
        hadamard(order, device=weight.device, dtype=weight.dtype)[:rows, :cols]
        / math.sqrt(order) #нормируем для стабильного обучения
    )

In [ ]:
@torch.no_grad()
def initialize(model: nn.Module, method: str) -> None:
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            if method == "zero":
                module.weight.zero_()
                center_h, center_w = module.kernel_size[0] // 2, module.kernel_size[1] // 2
                zero_matrix_(module.weight[:, :, center_h, center_w])
            elif method == "kaiming":
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif method == "xavier":
                nn.init.xavier_normal_(module.weight)
        elif isinstance(module, nn.Linear):
            if method == "zero":
                zero_matrix_(module.weight)
            elif method == "kaiming":
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            elif method == "xavier":
                nn.init.xavier_normal_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    if method == "zero":
        # зануляем последнюю свертку каждого скип-коннекшена на нули
        for module in model.modules():
            if isinstance(module, BasicBlock):
                nn.init.zeros_(module.conv2.weight)

In [ ]:
def stable_rank(weight: torch.Tensor) -> float:
    matrix = weight.detach().float().flatten(1)
    frobenius_sq = torch.sum(matrix * matrix)
    if frobenius_sq == 0:
        return 0.0
    spectral = torch.linalg.matrix_norm(matrix, ord=2)
    return (frobenius_sq / spectral.square()).item()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def make_loaders() -> tuple[DataLoader, DataLoader]:
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)
    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    test_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean, std)]
    )
    train_set = datasets.CIFAR10(
        DATA_DIR, train=True, download=True, transform=train_transform
    )
    test_set = datasets.CIFAR10(
        DATA_DIR, train=False, download=True, transform=test_transform
    )
    generator = torch.Generator().manual_seed(SEED)
    common = dict(
        batch_size=BATCH_SIZE,
        num_workers=WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=WORKERS > 0,
    )
    train_loader = DataLoader(
        train_set, shuffle=True, generator=generator, drop_last=True, **common
    )
    test_loader = DataLoader(test_set, shuffle=False, **common)
    return train_loader, test_loader

In [ ]:
def learning_rate(epoch: int) -> float:
    if epoch < WARMUP_EPOCHS:
        return LR * (epoch + 1) / WARMUP_EPOCHS
    drops = sum(epoch >= milestone for milestone in MILESTONES)
    return LR * (0.1**drops)

# Обучение

In [1]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None,
) -> tuple[float, float]:
    model.train(optimizer is not None)
    total_loss = total_correct = total = 0
    context = torch.enable_grad() if optimizer is not None else torch.inference_mode()
    with context:
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if optimizer is not None:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100.0 * total_correct / total


def tracked_convolutions(model: nn.Module) -> dict[str, nn.Conv2d]:
    names = ("layer2.0.conv1", "layer3.0.conv1") # "layer4.0.conv1" выключил, чтобы быстрее обучалось
    modules = dict(model.named_modules())
    return {name: modules[name] for name in names}

NameError: name 'nn' is not defined

In [ ]:
def train_one(
    method: str,
    train_loader: DataLoader,
    test_loader: DataLoader,
    device: torch.device,
) -> dict:
    set_seed(SEED)
    model = SmallResNet().to(device)
    initialize(model, method)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()
    tracked = tracked_convolutions(model)
    epochs = []
    test_errors = []
    rank_epochs = [0]
    ranks = {name: [stable_rank(conv.weight)] for name, conv in tracked.items()}
    best_accuracy = 0.0
    started = time.time()

    for epoch in range(EPOCHS):
        lr = learning_rate(epoch)
        for group in optimizer.param_groups:
            group["lr"] = lr
        _, train_accuracy = run_epoch(
            model, train_loader, criterion, device, optimizer
        )
        _, test_accuracy = run_epoch(
            model, test_loader, criterion, device, None
        )
        best_accuracy = max(best_accuracy, test_accuracy)
        epochs.append(epoch + 1)
        test_errors.append(100.0 - test_accuracy)
        if (epoch + 1) % RANK_EVERY == 0 or epoch + 1 == EPOCHS:
            rank_epochs.append(epoch + 1)
            for name, conv in tracked.items():
                ranks[name].append(stable_rank(conv.weight))
        print(
            f"{method:8s} {epoch + 1:3d}/{EPOCHS}: "
            f"lr={lr:.4g} train={train_accuracy:.2f}% "
            f"test={test_accuracy:.2f}% error={100 - test_accuracy:.2f}%"
        )

    print(
        f"{method}: final error={test_errors[-1]:.2f}%, "
        f"best error={100.0 - best_accuracy:.2f}%, "
        f"time={(time.time() - started) / 60:.1f} min"
    )
    return {
        "method": method,
        "epochs": epochs,
        "test_errors": test_errors,
        "rank_epochs": rank_epochs,
        "ranks": ranks,
    }

# Графики

In [5]:
def make_plots(runs: list[dict], output_dir: Path) -> None:
    colors = {"zero": "tab:blue", "kaiming": "tab:orange", "xavier": "tab:green"}
    output_dir.mkdir(parents=True, exist_ok=True)

    fig, axis = plt.subplots(figsize=(7, 4))
    for run in runs:
        method = run["method"]
        axis.plot(
            run["epochs"],
            run["test_errors"],
            label=method,
            color=colors[method],
        )
    axis.set(xlabel="Epoch", ylabel="Test error (%)", title="CIFAR-10 / Small ResNet")
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "test_error.png", dpi=160)
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharex=True)
    rank_names = ("layer2.0.conv1", "layer3.0.conv1")#, "layer4.0.conv1")
    for axis, rank_name in zip(axes, rank_names):
        for run in runs:
            method = run["method"]
            axis.plot(
                run["rank_epochs"],
                run["ranks"][rank_name],
                label=method,
                color=colors[method],
            )
        axis.set_title(rank_name)
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.25)
    axes[0].set_ylabel("Stable rank")
    axes[-1].legend()
    fig.tight_layout()
    fig.savefig(output_dir / "stable_rank_trajectories.png", dpi=160)
    plt.close(fig)

In [ ]:
if RANK_EVERY < 1:
    raise ValueError("RANK_EVERY must be >= 1")

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {device}")

runs = []
for method in METHODS:
    set_seed(SEED)
    train_loader, test_loader = make_loaders()
    runs.append(train_one(method, train_loader, test_loader, device))

make_plots(runs, Path(OUTPUT_DIR))